# CDCR Multi-Hazard Ranking

Ranks 30 active CDCR state prisons by overlapping mid-century climate hazard burden.

**Hazards:** heat, drought, flood, wildfire (FHSZ)  
**Method:** Each hazard re-normalized 0–1 (min-max) across the 30 facilities. Composite score = mean of 4 normalized values. Hazard count = number of hazards above per-hazard median.  
**Wildfire encoding:** Very High=3, High=2, Moderate=1, None=0, then normalized 0–1.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('../../')
DATA = ROOT / 'data'

In [2]:
# Load master facility-hazard file
all_fac = pd.read_csv(DATA / 'allfacilities_climate_hazards.csv')

# Load CDCR facility details for codes and city
cdcr = pd.read_csv(DATA / 'cdcr' / 'cdcr_facilities.csv')

# Load tract-level sub-components not in master file
flood_tracts = pd.read_csv(DATA / 'hazards' / 'flood_hazard.csv')
drought_tracts = pd.read_csv(DATA / 'hazards' / 'drought_hazard.csv')

print(f'All facilities: {len(all_fac)}')
print(f'CDCR rows: {len(cdcr)}')

All facilities: 357
CDCR rows: 84


In [3]:
# Filter to 30 active state prisons (have cdcr_code, exclude CRC)
cdcr_active = cdcr[
    (cdcr['cdcr_code'].notna()) & 
    (cdcr['cdcr_code'].str.strip() != '') &
    (cdcr['cdcr_code'] != 'CRC') &
    # Exclude facilities not in the heat risk index scope
    (~cdcr['cdcr_code'].isin(['CAC', 'FWF', 'CVSP']))
].copy()

print(f'Active CDCR state prisons: {len(cdcr_active)}')
assert len(cdcr_active) == 30, f'Expected 30, got {len(cdcr_active)}'

Active CDCR state prisons: 30


In [4]:
# Join hazard data from master file via facilityid
df = cdcr_active[['cdcr_code', 'name', 'city', 'facilityid', 'tract_geoid']].merge(
    all_fac[['facilityid', 
             'heat_hazard_midcentury_idx', 'heat_days_over_90_midcentury',
             'flood_hazard_midcentury_idx', 'flood_bam_500_pct',
             'drought_hazard_midcentury_idx',
             'fire_fhsz']],
    on='facilityid', how='left'
)

# Join tract-level sub-components
df['tract_geoid'] = df['tract_geoid'].astype(str).str.zfill(11)
flood_tracts['GEOID'] = flood_tracts['GEOID'].astype(str).str.zfill(11)
drought_tracts['GEOID'] = drought_tracts['GEOID'].astype(str).str.zfill(11)

df = df.merge(
    flood_tracts[['GEOID', 'flood_verywet_fut_pct']],
    left_on='tract_geoid', right_on='GEOID', how='left'
).drop(columns='GEOID')

df = df.merge(
    drought_tracts[['GEOID', 'Dr_WSV_average']],
    left_on='tract_geoid', right_on='GEOID', how='left'
).drop(columns='GEOID')

print(f'Joined rows: {len(df)}')
df[['cdcr_code', 'heat_hazard_midcentury_idx', 'drought_hazard_midcentury_idx', 
    'flood_hazard_midcentury_idx', 'fire_fhsz']].head(10)

Joined rows: 30


,cdcr_code,heat_hazard_midcentury_idx,drought_hazard_midcentury_idx,flood_hazard_midcentury_idx,fire_fhsz
0,CAL,70.593748,33.101490,10.586326,Moderate
1,CEN,79.967830,18.680458,16.225309,NaN
2,SOL,49.408014,48.140138,27.196745,Moderate
3,ASP,58.799240,30.416199,27.299888,NaN
4,CMC,10.213356,35.749720,30.822899,Moderate
5,COR,87.315082,61.846664,38.674526,NaN
6,SATF,87.315082,61.846664,38.674526,NaN
7,CCWF,74.187427,78.417307,22.152963,NaN
8,RJD,47.570086,33.724379,18.562565,Very High
9,PBSP,8.781720,13.112750,28.823454,Moderate


In [5]:
# Encode wildfire FHSZ as ordinal
fhsz_map = {'Very High': 3, 'High': 2, 'Moderate': 1}
df['fire_fhsz_ordinal'] = df['fire_fhsz'].map(fhsz_map).fillna(0)

print('FHSZ distribution:')
print(df['fire_fhsz'].fillna('None').value_counts())

FHSZ distribution:
fire_fhsz
None         14
Moderate     11
Very High     3
High          2
Name: count, dtype: int64


In [6]:
# Min-max normalize each hazard across the 30 facilities
def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

df['heat_norm'] = minmax(df['heat_hazard_midcentury_idx'])
df['drought_norm'] = minmax(df['drought_hazard_midcentury_idx'])
df['flood_norm'] = minmax(df['flood_hazard_midcentury_idx'])
df['fire_norm'] = minmax(df['fire_fhsz_ordinal'])

# Composite score: mean of 4 normalized hazards
df['multi_hazard_score'] = df[['heat_norm', 'drought_norm', 'flood_norm', 'fire_norm']].mean(axis=1)

# Hazard count: number above per-hazard median
hazard_cols = ['heat_norm', 'drought_norm', 'flood_norm', 'fire_norm']
medians = df[hazard_cols].median()
print('Per-hazard medians (normalized):')
print(medians.round(3))

df['hazard_count'] = sum(df[col] > medians[col] for col in hazard_cols).astype(int)

# Text list of which hazards are above median
hazard_labels = {'heat_norm': 'heat', 'drought_norm': 'drought', 'flood_norm': 'flood', 'fire_norm': 'wildfire'}
df['hazards_above_median'] = df.apply(
    lambda r: ', '.join(hazard_labels[c] for c in hazard_cols if r[c] > medians[c]), axis=1
)

# Formatted facility label: "Name (CODE)\nCity, CA"
df['facility_label'] = df['name'] + ' (' + df['cdcr_code'] + ')\n' + df['city'] + ', CA'

Per-hazard medians (normalized):
heat_norm       0.618
drought_norm    0.536
flood_norm      0.246
fire_norm       0.333
dtype: float64


In [7]:
# Build final output table
output = df.sort_values('multi_hazard_score', ascending=False).reset_index(drop=True)
output.index += 1  # 1-based rank
output.index.name = 'rank'

output = output.rename(columns={
    'cdcr_code': 'cdcr_code',
    'name': 'facility_name',
    'city': 'city',
    'facility_label': 'facility_label',
    'multi_hazard_score': 'multi_hazard_score',
    'hazard_count': 'hazard_count',
    'hazards_above_median': 'hazards_above_median',
    'heat_hazard_midcentury_idx': 'heat_raw',
    'heat_norm': 'heat_norm',
    'heat_days_over_90_midcentury': 'heat_days_over_90f',
    'drought_hazard_midcentury_idx': 'drought_raw',
    'drought_norm': 'drought_norm',
    'Dr_WSV_average': 'drought_wsv',
    'flood_hazard_midcentury_idx': 'flood_raw',
    'flood_norm': 'flood_norm',
    'flood_bam_500_pct': 'flood_500yr_floodplain_pct',
    'flood_verywet_fut_pct': 'flood_verywet_pct',
    'fire_fhsz': 'wildfire_fhsz',
    'fire_norm': 'wildfire_norm',
})

out_cols = [
    'cdcr_code', 'facility_name', 'city', 'facility_label',
    'multi_hazard_score', 'hazard_count', 'hazards_above_median',
    'heat_raw', 'heat_norm', 'heat_days_over_90f',
    'drought_raw', 'drought_norm', 'drought_wsv',
    'flood_raw', 'flood_norm', 'flood_500yr_floodplain_pct', 'flood_verywet_pct',
    'wildfire_fhsz', 'wildfire_norm',
]
output = output[out_cols]

print(f'Output shape: {output.shape}')
output.head(10)

Output shape: (30, 19)


,cdcr_code,facility_name,city,facility_label,multi_hazard_score,hazard_count,hazards_above_median,heat_raw,heat_norm,heat_days_over_90f,drought_raw,drought_norm,drought_wsv,flood_raw,flood_norm,flood_500yr_floodplain_pct,flood_verywet_pct,wildfire_fhsz,wildfire_norm
rank,,,,,,,,,,,,,,,,,,,
1,LAC,California State Prison-Los Angeles County,Lancaster,California State Prison-Los Angeles County (LA...,0.834192,4,"heat, drought, flood, wildfire",74.528425,0.776087,125.362366,49.727829,0.560682,27.609209,82.314628,1.000000,100.000000,21.305040,Very High,1.000000
2,SCC,Sierra Conservation Center,Jamestown,"Sierra Conservation Center (SCC)\nJamestown, CA",0.643532,3,"heat, drought, wildfire",61.817272,0.626042,117.484406,63.636684,0.773666,41.774831,14.357172,0.174418,0.000000,21.913870,Very High,1.000000
3,CIW,California Institution For Women,Corona,California Institution For Women (CIW)\nCorona...,0.580345,3,"heat, flood, wildfire",72.105937,0.747492,115.174580,45.621904,0.497808,30.226712,33.700797,0.409414,9.652510,30.203413,High,0.666667
4,MCSP,Mule Creek State Prison,Ione,"Mule Creek State Prison (MCSP)\nIone, CA",0.539912,2,"drought, wildfire",53.559888,0.528571,118.186913,63.695132,0.774561,45.769790,15.627446,0.189850,2.654867,21.549051,High,0.666667
5,COR,"California State Prison, Corcoran",Corcoran,"California State Prison, Corcoran (COR)\nCorco...",0.535779,3,"heat, drought, flood",87.315082,0.927024,141.902024,61.846664,0.746256,44.597172,38.674526,0.469838,32.631579,23.113828,NaN,0.000000
6,SATF,Ca Substance Abuse Treatment Facility,Corcoran,Ca Substance Abuse Treatment Facility (SATF)\n...,0.535779,3,"heat, drought, flood",87.315082,0.927024,141.902024,61.846664,0.746256,44.597172,38.674526,0.469838,32.631579,23.113828,NaN,0.000000
7,CCWF,Central California Women'S Facility,Chowchilla,Central California Women'S Facility (CCWF)\nCh...,0.510297,3,"heat, drought, flood",74.187427,0.772062,130.767222,78.417307,1.000000,58.292690,22.152963,0.269125,12.237094,21.499645,NaN,0.000000
8,VSP,Valley State Prison,Chowchilla,"Valley State Prison (VSP)\nChowchilla, CA",0.510297,3,"heat, drought, flood",74.187427,0.772062,130.767222,78.417307,1.000000,58.292690,22.152963,0.269125,12.237094,21.499645,NaN,0.000000
9,RJD,R J Donovan Correctional Facility,San Diego,R J Donovan Correctional Facility (RJD)\nSan D...,0.499749,1,wildfire,47.570086,0.457866,29.283005,33.724379,0.315623,27.883503,18.562565,0.225507,0.000000,24.660104,Very High,1.000000


In [8]:
# Full table
output.round(3)

,cdcr_code,facility_name,city,facility_label,multi_hazard_score,hazard_count,hazards_above_median,heat_raw,heat_norm,heat_days_over_90f,drought_raw,drought_norm,drought_wsv,flood_raw,flood_norm,flood_500yr_floodplain_pct,flood_verywet_pct,wildfire_fhsz,wildfire_norm
rank,,,,,,,,,,,,,,,,,,,
1,LAC,California State Prison-Los Angeles County,Lancaster,California State Prison-Los Angeles County (LA...,0.834,4,"heat, drought, flood, wildfire",74.528,0.776,125.362,49.728,0.561,27.609,82.315,1.000,100.000,21.305,Very High,1.000
2,SCC,Sierra Conservation Center,Jamestown,"Sierra Conservation Center (SCC)\nJamestown, CA",0.644,3,"heat, drought, wildfire",61.817,0.626,117.484,63.637,0.774,41.775,14.357,0.174,0.000,21.914,Very High,1.000
3,CIW,California Institution For Women,Corona,California Institution For Women (CIW)\nCorona...,0.580,3,"heat, flood, wildfire",72.106,0.747,115.175,45.622,0.498,30.227,33.701,0.409,9.653,30.203,High,0.667
4,MCSP,Mule Creek State Prison,Ione,"Mule Creek State Prison (MCSP)\nIone, CA",0.540,2,"drought, wildfire",53.560,0.529,118.187,63.695,0.775,45.770,15.627,0.190,2.655,21.549,High,0.667
5,COR,"California State Prison, Corcoran",Corcoran,"California State Prison, Corcoran (COR)\nCorco...",0.536,3,"heat, drought, flood",87.315,0.927,141.902,61.847,0.746,44.597,38.675,0.470,32.632,23.114,NaN,0.000
6,SATF,Ca Substance Abuse Treatment Facility,Corcoran,Ca Substance Abuse Treatment Facility (SATF)\n...,0.536,3,"heat, drought, flood",87.315,0.927,141.902,61.847,0.746,44.597,38.675,0.470,32.632,23.114,NaN,0.000
7,CCWF,Central California Women'S Facility,Chowchilla,Central California Women'S Facility (CCWF)\nCh...,0.510,3,"heat, drought, flood",74.187,0.772,130.767,78.417,1.000,58.293,22.153,0.269,12.237,21.500,NaN,0.000
8,VSP,Valley State Prison,Chowchilla,"Valley State Prison (VSP)\nChowchilla, CA",0.510,3,"heat, drought, flood",74.187,0.772,130.767,78.417,1.000,58.293,22.153,0.269,12.237,21.500,NaN,0.000
9,RJD,R J Donovan Correctional Facility,San Diego,R J Donovan Correctional Facility (RJD)\nSan D...,0.500,1,wildfire,47.570,0.458,29.283,33.724,0.316,27.884,18.563,0.226,0.000,24.660,Very High,1.000


In [9]:
# Save CSV
output.to_csv('CDCR_multi_hazard_rank.csv')
print('Saved CDCR_multi_hazard_rank.csv')

Saved CDCR_multi_hazard_rank.csv


In [10]:
# Summary stats
print('Hazard count distribution:')
print(output['hazard_count'].value_counts().sort_index())
print(f'\nTop 5 multi-hazard facilities:')
print(output[['cdcr_code', 'city', 'multi_hazard_score', 'hazard_count']].head())
print(f'\nBottom 5:')
print(output[['cdcr_code', 'city', 'multi_hazard_score', 'hazard_count']].tail())

Hazard count distribution:
hazard_count
0     5
1    10
2     6
3     8
4     1
Name: count, dtype: int64

Top 5 multi-hazard facilities:
     cdcr_code       city  multi_hazard_score  hazard_count
rank                                                       
1          LAC  Lancaster            0.834192             4
2          SCC  Jamestown            0.643532             3
3          CIW     Corona            0.580345             3
4         MCSP       Ione            0.539912             2
5          COR   Corcoran            0.535779             3

Bottom 5:
     cdcr_code             city  multi_hazard_score  hazard_count
rank                                                             
26        SVSP          Soledad            0.292827             0
27         CEN         Imperial            0.280667             1
28          SQ      San Quentin            0.274555             0
29         CMC  San Luis Obispo            0.267830             1
30        PBSP    Crescent City    